In [1]:
import os

import matplotlib.pyplot as plt
from matplotlib_venn import venn3, venn3_circles
from matplotlib.colors import LinearSegmentedColormap

import numpy as np
import pandas as pd

import re

import squarify

from slugify import slugify

# Get the root_path for this jupyter notebook repo.
repo_path = os.path.dirname(os.path.abspath(os.getcwd()))
col_config_path = os.path.join(
    repo_path, 'files', 'IMLS-FAIR-CARE-Survey', 'imls-fair-care-survey-columns-config.csv',
)
select_figs_path = os.path.join(
    repo_path, 'files', 'IMLS-FAIR-CARE-Survey', 'figs-select-options',
)

processed_survey_path = '/home/ekansa/oc-data/fair-care-survey-processed.csv' # Keep this OUT of version control, has sensitive info
wide_survey_path = '/home/ekansa/oc-data/fair-care-survey-multi-select-expanded.csv' # Keep this OUT of version control, has sensitive info
col_summary_path = '/home/ekansa/oc-data/fair-care-survey-col-summary.csv'

df_config = pd.read_csv(col_config_path, low_memory=False)
df_orig = pd.read_csv(processed_survey_path, low_memory=False)
df_col_sum = pd.read_csv(col_summary_path, low_memory=False)
df_wide = pd.read_csv(wide_survey_path, low_memory=False)

print(f'FAIR+CARE survey has {len(df_orig.index)} rows')
print(f'FAIR+CARE survey derived multi-select -wide- has {len(df_wide.index)} rows and {len(df_wide.columns)} columns')
print(f'FAIR+CARE survey column summary has {len(df_col_sum.index)} rows')


FAIR+CARE survey has 787 rows
FAIR+CARE survey derived multi-select -wide- has 787 rows and 271 columns
FAIR+CARE survey column summary has 151 rows


In [2]:
def get_original_column_metadata(column, df_config=df_config):
    """Gets the original column name from the df_config (column configuration data)"""
    config_index = (df_config['Working_Column_Name'] == column)
    if len(df_config[config_index].index) != 1:
        # We didn't find a matching column name
        return None
    row = df_config[config_index].iloc[0]
    keys = ['column', 'orig_column_index', 'Question Number', 'Section', 'Sub-Section', 'raw_column',]
    col_metadata = {k: row[k] for k in keys if k in df_config.columns.tolist()}
    return col_metadata


def get_select_option_value_from_tf_col(tf_col):
    """Gets the select option value from a TF column"""
    col_split = tf_col.split('::')
    select_option_value = col_split[1] # The second element.
    return select_option_value


def insert_line_breaks(text, max_chars):
    """Insert line breaks for long text"""
    words = text.split()  # Split the text into words
    current_line = []
    result = []
    for word in words:
        # Check if adding the next word exceeds the max_chars limit
        if len(' '.join(current_line + [word])) > max_chars:
            # Join the current line and add it to the result list
            result.append(' '.join(current_line))
            # Start a new line with the current word
            current_line = [word]
        else:
            # Add the word to the current line
            current_line.append(word)
    # Add the last line to the result list
    if current_line:
        result.append(' '.join(current_line))
    # Join all lines with a newline character
    return '\n'.join(result)



def create_barchart(
    sel_col, 
    df_wide, 
    all_sel_col_option_sorts=None, 
    filter_index=None,
    title_suffix='',
    file_suffix='',
    save_dir=select_figs_path,
):
    """Make a barchart plot for a given selection column"""
    if not sel_col in df_wide.columns.tolist():
        # This columns doesn't exist
        return all_sel_col_option_sorts
    # Get metadata for the column, needed for captions and titles
    col_metadata =  get_original_column_metadata(sel_col)
    if not col_metadata:
        # We don't have metadata for this column for some reason!
        return all_sel_col_option_sorts

    if filter_index is None:
        # We haven't set a filter index so make one that selects everything (default)
        filter_index = ~df_wide['Response ID'].isnull()

    # Now limit the filter to include only responses that are not null for this sel_column.
    sel_col_index = filter_index & ~df_wide[sel_col].isnull()
    # Get the count of the total number of responses for this column
    count_total_response = len(df_wide[sel_col_index].index)
    if not count_total_response:
        # We don't have data for the column!
        return all_sel_col_option_sorts
    tf_cols = [c for c in df_wide.columns.tolist() if c.startswith(sel_col) and c.endswith('::TF')]
    if not tf_cols:
        # we don't have T/F columns for the column!
        return all_sel_col_option_sorts
    if not all_sel_col_option_sorts:
        all_sel_col_option_sorts = {}

    # Get TF selection columns in their sorted order, if this sorted order exists
    sorted_tf_cols = all_sel_col_option_sorts.get(sel_col)
    
    if not sorted_tf_cols:
        # We don't have  TF selection columns in their sorted order, so
        # sort them based on their frequency.
        tf_col_size_tups = []
        for tf_col in tf_cols:
            tf_col_index = sel_col_index & (df_wide[tf_col] == True)
            select_option_count = len(df_wide[tf_col_index].index)
            tf_col_size_tups.append((tf_col, select_option_count))
        # Sort the tf_cols and sizes in reverse order
        sorted_tf_col_size_tups = sorted(tf_col_size_tups, key=lambda x: x[1], reverse=True)
        sorted_tf_cols = [c for c,_ in sorted_tf_col_size_tups]
        all_sel_col_option_sorts[sel_col] = sorted_tf_cols

    # OK! Now we have our TF selection columns in their sorted order
    categories = []
    values = []
    for tf_col in sorted_tf_cols:
        select_option_value = get_select_option_value_from_tf_col(tf_col)
        select_option_value = insert_line_breaks(select_option_value, 40)
        categories.append(select_option_value)
        tf_col_index = sel_col_index & (df_wide[tf_col] == True)
        select_option_count = len(df_wide[tf_col_index].index)
        values.append(select_option_count)

    # OK Now we can build our plot.
    plt.rcParams["figure.figsize"] = (6, 6)
    bars = plt.bar(categories, values)
    base_color = plt.cm.Blues(0.5)
    faded_colors = [plt.cm.Blues(1 - (i/(1.5 * len(categories)))) for i in range(len(categories))]
    for bar, color in zip(bars, faded_colors):
        bar.set_color(color)
        
    section_part = ''
    question_part = ''
    question_part_cap = ''
    if col_metadata["Section"] and col_metadata["Sub-Section"]:
        section_part = f'{col_metadata["Section"]} -- {col_metadata["Sub-Section"]} '
    if col_metadata["Question Number"] > 0:
        question_num = int(col_metadata["Question Number"])
        question_part = f'({question_num}) '
        question_part_cap = f'[Question: {question_num}] '
    title = f'{section_part}{question_part}{sel_col}'

    # Add line breaks in the raw_column to make formatting easier.
    raw_column = insert_line_breaks(col_metadata["raw_column"], 60)
    caption = f'{question_part_cap}"{raw_column}"\n(Question had n = {count_total_response} total responses)'

    plt.title(f'{title}{title_suffix}')
    plt.xlabel(caption, fontsize=9)
    plt.xticks(rotation=45, ha='right', fontsize=6)
    
    slug_file = slugify(f'bar-{title}')
    plt.tight_layout()
    plt.autoscale()
    filename = f'{slug_file}{file_suffix}.png'
    f_path = os.path.join(save_dir, filename)
    plt.savefig(f_path, bbox_inches='tight', dpi=150)
    plt.close()
    print(f'Saved figure: {filename}')
    # Return our dictionary to keep the TF selection columns
    # sorting consistent.
    return all_sel_col_option_sorts

In [3]:
# Make barcharts for all the responses, with no filters, for all multi-select columns in the entire dataset.
all_sel_col_option_sorts = None
for sel_col in df_wide.columns.tolist():
    all_sel_col_option_sorts = create_barchart(sel_col, df_wide, all_sel_col_option_sorts=all_sel_col_option_sorts)

# Now make these plots with various filters.

Saved figure: bar-demographics-general-demo-race-ethnicity.png
Saved figure: bar-demographics-general-work-work-setting.png
Saved figure: bar-demographics-general-experience-data.png
Saved figure: bar-demographics-general-region-focus-work-research-region.png
Saved figure: bar-demographics-data-role-1-data-role-how-you-work-with-data.png
Saved figure: bar-fair-findable-3-data-store-data-storage.png
Saved figure: bar-fair-findable-5-identifiers-identifier-types.png
Saved figure: bar-fair-findable-7-metadata-search-metadata-search-method.png
Saved figure: bar-fair-findable-8-supplemental-data-data-types.png
Saved figure: bar-fair-findable-9-cost-paid-response.png
Saved figure: bar-fair-accessible-10-accessible-access-type.png
Saved figure: bar-fair-accessible-11-why-acquire-data-acquisition.png
Saved figure: bar-fair-accessible-13-attribute-how-do-you-attribute-data.png
Saved figure: bar-fair-interoperable-15-shared-terms-shared-terms-that-best-work.png
Saved figure: bar-fair-reusable-16

In [4]:


filtered_plot_configs = [
    # (filter_index, title_suffix, file_suffix,),
    ((df_wide['Work: Work setting::Academic: University or College::TF'] == True), '\nAcademic (Univ)', '-academic',),
    ((df_wide['Work: Work setting::Academic: University or College::TF'] == False), '\nNot Academic (Univ)', '-not-academic',),
    ((df_wide['Region Focus: Work/research Region::North America (specify region)::TF'] == True), '\nN. America Focus', '-n-america',),
    ((df_wide['Region Focus: Work/research Region::North America (specify region)::TF'] == False), '\nOutside N. America Focus', '-out-n-america',),
    ((df_wide['Response Type: Individual or Org '] == 'Individual'), '\nResponding as Individual', '-resp-ind',),
    ((df_wide['Response Type: Individual or Org '].str.contains('Organization')), '\nResponding as Organization', '-resp-org',),
]
for sel_col in df_wide.columns.tolist():
    for filter_index, title_suffix, file_suffix in filtered_plot_configs:
        all_sel_col_option_sorts = create_barchart(
            sel_col, 
            df_wide, 
            all_sel_col_option_sorts=all_sel_col_option_sorts,
            filter_index=filter_index,
            title_suffix=title_suffix,
            file_suffix=file_suffix,
        )


Saved figure: bar-demographics-general-demo-race-ethnicity-academic.png
Saved figure: bar-demographics-general-demo-race-ethnicity-not-academic.png
Saved figure: bar-demographics-general-demo-race-ethnicity-n-america.png
Saved figure: bar-demographics-general-demo-race-ethnicity-out-n-america.png
Saved figure: bar-demographics-general-demo-race-ethnicity-resp-ind.png
Saved figure: bar-demographics-general-demo-race-ethnicity-resp-org.png
Saved figure: bar-demographics-general-work-work-setting-academic.png
Saved figure: bar-demographics-general-work-work-setting-not-academic.png
Saved figure: bar-demographics-general-work-work-setting-n-america.png
Saved figure: bar-demographics-general-work-work-setting-out-n-america.png
Saved figure: bar-demographics-general-work-work-setting-resp-ind.png
Saved figure: bar-demographics-general-work-work-setting-resp-org.png
Saved figure: bar-demographics-general-experience-data-academic.png
Saved figure: bar-demographics-general-experience-data-not-a

In [5]:
df_wide.columns.tolist()

['Response ID',
 'Demo: Age Range',
 'Demo: Gender',
 'Demo: Race/Ethnicity',
 'Demo: Race/Ethnicity::White or Caucasian::TF',
 'Demo: Race/Ethnicity::Asian::TF',
 'Demo: Race/Ethnicity::Native Hawaiian or Other Pacific Islander::TF',
 'Demo: Race/Ethnicity::Black or African American::TF',
 'Demo: Race/Ethnicity::Other::TF',
 'Demo: Race/Ethnicity::American Indian / Native American or Alaska Native::TF',
 'Edu: Highest Degree',
 'Work: Work setting',
 'Work: Work setting::Academic: University or College::TF',
 'Work: Work setting::Non-governmental Organization::TF',
 'Work: Work setting::Museum: Public/Academic::TF',
 'Work: Work setting::Academic: Community College::TF',
 'Work: Work setting::Library: Tribal::TF',
 'Work: Work setting::Archive or Digital Repository::TF',
 'Work: Work setting::CRM: Cultural Resources Consulting Firm::TF',
 'Work: Work setting::Library: Academic::TF',
 'Work: Work setting::CRM: Tribally-Owned or similar Consulting Firm::TF',
 'Work: Work setting::Archiv